# ChatGPT Prompt Dataset Exploration

This notebook provides a comprehensive exploration of the ChatGPT prompt dataset, including:
- Dataset structure and overview
- Prompt distribution across categories
- Tag analysis and frequency
- Use case categorization
- Difficulty level assessment

In [ ]:
# Import required libraries
import json
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 1. Load Dataset

In [ ]:
# Define the data directory path
DATA_DIR = Path('../chatgpt-prompt-dataset/data')

# List all JSON files
json_files = list(DATA_DIR.glob('*.json'))
print(f"Found {len(json_files)} JSON files:")
for f in json_files:
    print(f"  - {f.name}")

In [ ]:
def load_prompt_dataset(file_path):
    """Load a single prompt dataset JSON file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_all_datasets(data_dir):
    """Load all prompt datasets from the data directory."""
    datasets = {}
    for file_path in data_dir.glob('*.json'):
        data = load_prompt_dataset(file_path)
        datasets[data['category']] = data
    return datasets

# Load all datasets
all_datasets = load_all_datasets(DATA_DIR)
print(f"\nLoaded {len(all_datasets)} datasets: {list(all_datasets.keys())}")

## 2. Dataset Overview

In [ ]:
# Create a summary dataframe
summary_data = []
for category, data in all_datasets.items():
    summary_data.append({
        'Category': category,
        'Level': data.get('level', 'N/A'),
        'Description': data.get('description', 'N/A')[:60] + '...',
        'Prompt Count': len(data.get('prompts', []))
    })

summary_df = pd.DataFrame(summary_data)
print("Dataset Summary:")
display(summary_df)
print(f"\nTotal Prompts: {summary_df['Prompt Count'].sum()}")

In [ ]:
# Convert all prompts to a single DataFrame
all_prompts = []
for category, data in all_datasets.items():
    for prompt in data.get('prompts', []):
        prompt['category'] = category
        prompt['level'] = data.get('level', 'N/A')
        all_prompts.append(prompt)

prompts_df = pd.DataFrame(all_prompts)
print(f"Combined DataFrame shape: {prompts_df.shape}")
print(f"\nColumns: {list(prompts_df.columns)}")
prompts_df.head()

## 3. Prompt Distribution Analysis

In [ ]:
# Plot prompt count by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
category_counts = prompts_df['category'].value_counts()
bars = axes[0].bar(category_counts.index, category_counts.values, color=sns.color_palette('husl', len(category_counts)))
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Number of Prompts', fontsize=12)
axes[0].set_title('Prompt Distribution by Category', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, count in zip(bars, category_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                  str(count), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('husl', len(category_counts)), startangle=90)
axes[1].set_title('Prompt Distribution Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Level distribution
fig, ax = plt.subplots(figsize=(8, 5))
level_counts = prompts_df['level'].value_counts()
bars = ax.bar(level_counts.index, level_counts.values, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'][:len(level_counts)])
ax.set_xlabel('Level', fontsize=12)
ax.set_ylabel('Number of Prompts', fontsize=12)
ax.set_title('Prompt Distribution by Difficulty Level', fontsize=14, fontweight='bold')

for bar, count in zip(bars, level_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Tag Analysis

In [ ]:
# Extract all tags
all_tags = []
for tags in prompts_df['tags']:
    if isinstance(tags, list):
        all_tags.extend(tags)

# Count tag frequencies
tag_counts = Counter(all_tags)
print(f"Total unique tags: {len(tag_counts)}")
print(f"\nTop 20 most common tags:")
for tag, count in tag_counts.most_common(20):
    print(f"  {tag}: {count}")

In [ ]:
# Visualize top tags
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 tags bar chart
top_tags = dict(tag_counts.most_common(15))
bars = axes[0].barh(list(top_tags.keys())[::-1], list(top_tags.values())[::-1], 
                    color=sns.color_palette('viridis', len(top_tags)))
axes[0].set_xlabel('Frequency', fontsize=12)
axes[0].set_title('Top 15 Most Common Tags', fontsize=14, fontweight='bold')

# Word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white', 
                      colormap='viridis', max_words=50).generate_from_frequencies(tag_counts)
axes[1].imshow(wordcloud, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Tag Word Cloud', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Tags by category heatmap
def get_tags_by_category(df):
    """Create a matrix of tag counts by category."""
    tag_category_matrix = {}
    for _, row in df.iterrows():
        category = row['category']
        tags = row['tags'] if isinstance(row['tags'], list) else []
        for tag in tags:
            if tag not in tag_category_matrix:
                tag_category_matrix[tag] = {}
            tag_category_matrix[tag][category] = tag_category_matrix[tag].get(category, 0) + 1
    return pd.DataFrame(tag_category_matrix).fillna(0).T

tag_matrix = get_tags_by_category(prompts_df)
# Show top 20 tags
top_tag_names = [t[0] for t in tag_counts.most_common(20)]
tag_matrix_top = tag_matrix.loc[tag_matrix.index.isin(top_tag_names)]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(tag_matrix_top, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Tag Distribution by Category (Top 20 Tags)', fontsize=14, fontweight='bold')
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Tag', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Use Case Analysis

In [ ]:
# Extract use cases
use_cases = prompts_df['use_case'].value_counts()
print(f"Total unique use cases: {len(use_cases)}")
print(f"\nTop 15 use cases:")
for use_case, count in use_cases.head(15).items():
    print(f"  {use_case}: {count}")

In [ ]:
# Visualize use case distribution
fig, ax = plt.subplots(figsize=(12, 8))
top_use_cases = use_cases.head(20)
bars = ax.barh(top_use_cases.index[::-1], top_use_cases.values[::-1], 
               color=sns.color_palette('coolwarm', len(top_use_cases)))
ax.set_xlabel('Count', fontsize=12)
ax.set_title('Top 20 Use Cases', fontsize=14, fontweight='bold')

for bar, count in zip(bars, top_use_cases.values[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
            str(count), va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Prompt Length Analysis

In [ ]:
# Calculate prompt lengths
prompts_df['prompt_length'] = prompts_df['prompt'].apply(len)
prompts_df['word_count'] = prompts_df['prompt'].apply(lambda x: len(x.split()))

print("Prompt Length Statistics:")
print(prompts_df[['prompt_length', 'word_count']].describe())

In [ ]:
# Prompt length distribution by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
categories = prompts_df['category'].unique()
data_by_category = [prompts_df[prompts_df['category'] == cat]['word_count'].values for cat in categories]
bp = axes[0].boxplot(data_by_category, labels=categories, patch_artist=True)
colors = sns.color_palette('husl', len(categories))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Word Count', fontsize=12)
axes[0].set_title('Prompt Length Distribution by Category', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Histogram
for cat in categories:
    subset = prompts_df[prompts_df['category'] == cat]['word_count']
    axes[1].hist(subset, alpha=0.5, label=cat, bins=15)
axes[1].set_xlabel('Word Count', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Word Count Histogram by Category', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Sample Prompts by Category

In [ ]:
# Display sample prompts from each category
def display_sample_prompts(df, category, n=3):
    """Display sample prompts from a specific category."""
    samples = df[df['category'] == category].sample(min(n, len(df[df['category'] == category])))
    print(f"\n{'='*60}")
    print(f"Category: {category.upper()}")
    print('='*60)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n[{i}] {row['title']}")
        print(f"    Prompt: {row['prompt'][:150]}..." if len(row['prompt']) > 150 else f"    Prompt: {row['prompt']}")
        print(f"    Use Case: {row['use_case']}")
        print(f"    Tags: {', '.join(row['tags'])}")

for category in prompts_df['category'].unique():
    display_sample_prompts(prompts_df, category, n=2)

## 8. Export Summary Statistics

In [ ]:
# Create and export summary statistics
summary_stats = {
    'total_prompts': len(prompts_df),
    'total_categories': len(prompts_df['category'].unique()),
    'total_unique_tags': len(tag_counts),
    'total_unique_use_cases': len(use_cases),
    'avg_prompt_length': prompts_df['word_count'].mean(),
    'category_distribution': prompts_df['category'].value_counts().to_dict(),
    'level_distribution': prompts_df['level'].value_counts().to_dict(),
    'top_10_tags': dict(tag_counts.most_common(10))
}

print("Summary Statistics:")
print(json.dumps(summary_stats, indent=2))

# Save summary
with open('dataset_summary.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)

print("\nSummary saved to dataset_summary.json")

## 9. Conclusions

This exploration reveals:

1. **Dataset Composition**: The dataset contains prompts across 6 categories with varying complexity levels
2. **Tag Diversity**: A rich set of tags enables flexible prompt categorization and search
3. **Use Case Coverage**: Prompts address diverse real-world applications
4. **Length Variation**: Prompt complexity varies significantly by category

Next steps: See `prompt_analysis.ipynb` for deeper pattern analysis.